# Run Gnani Evon v3.3 on Google Colab (free T4) with Ollama

This is the setup MachineMitra's live demo uses.

1. **Runtime → Change runtime type → T4 GPU.**
2. Run the cells one at a time.
3. Paste the four lines printed by the last cell into your terminal (or Streamlit secrets).

It uses the community `IQ4_XS` quantization of Evon v3.3 (about 18 GB), and about 97% of it runs on the T4.
Thinking is switched off through Ollama's native API, and the app streams answers so the free tunnel
never times out. Keep this tab open while you demo: if Colab stops, the app falls back to labelled rules.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!apt-get -qq update && apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import os, subprocess, time
os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
subprocess.Popen(["ollama", "serve"], stdout=open("/tmp/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
TAG = "hf.co/luminenio/gnani-evon-v3.3-30B-A3B-GGUF:IQ4_XS"
!ollama pull {TAG}

In [ ]:
# keep Evon loaded on the GPU, then a quick test with thinking off
import requests
requests.post("http://localhost:11434/api/generate", json={"model": TAG, "keep_alive": -1}, timeout=600)
r = requests.post("http://localhost:11434/api/chat", json={"model": TAG, "stream": False, "think": False,
    "messages": [{"role": "user", "content": "CNC machine ka spindle garam ho raha hai. Short answer."}]}, timeout=600)
print(r.json()["message"]["content"])
!ollama ps

In [ ]:
import re
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared && chmod +x /tmp/cloudflared
subprocess.Popen(["/tmp/cloudflared", "tunnel", "--no-autoupdate", "--url", "http://localhost:11434",
                  "--http-host-header", "localhost:11434"], stdout=open("/tmp/t.log", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/tmp/t.log").read())
    if m: url = m.group(0); break
    time.sleep(2)
print(f'$env:EVON_BASE_URL="{url}"')
print(f'$env:EVON_MODEL="{TAG}"')
print('$env:EVON_API="ollama"')
print('$env:EVON_TIMEOUT="300"')